# Choose what to try next

A small Python loop can propose a revision, test it, and keep the better version.
That is often enough. Meta-Evolve becomes useful when you want to compare search
choices, inspect every attempt, or keep the history.

**Start with a version → revise it → test it → inspect what happened.**

A **search policy** chooses which version to revise next. Here, following the
best version gets stuck at score **5**; exploring another branch reaches **10**
with the same seven evaluations. We'll run both and see why.

This is a deliberately constructed search demonstration with named versions
and assigned scores, not an empirical optimizer benchmark. It needs no model,
API key, checkout, or earlier lesson.



<a id="set-up-this-page"></a>
<a id="complete-shared-source"></a>

## 1. Install

Use a fresh notebook environment running **Python 3.12 or newer**.
Install directly from the published documentation:

In [ ]:
%pip install https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip

If you already imported Meta-Evolve, restart the kernel after installing.
Then run the remaining cells in order.

**Archived or offline docs:** use the ZIP included with that build. Put
`meta-evolve.zip` in the notebook's working folder (`%pwd` shows it; hosted
notebooks let you upload files), then run `%pip install ./meta-evolve.zip`
instead. Installing from source may still download build tools.

<a id="2-define-a-parser-and-measure-it"></a>

## 2. Start with versions and scores

Think of each name as a possible version of your prompt, skill, or program.
These numbers make the choice easy to see:

```text
seed (0)
├─ attractive (5) → plateau versions (5)
├─ delayed (1)    → winner (10)
└─ weak versions (0)
```

The **evaluator** measures a version. Here, `evaluate` simply looks up its
score; in your task it would run your checks. `Task` groups that evaluator
and the work allowance: six proposed revisions, plus one evaluation of the
starting version. Higher scores win.

In [ ]:
import meta_evolve as meta


MAX_TRIALS = 6
ROOT_SEED = 11
RUN_BUDGET = meta.Budget(evaluations=MAX_TRIALS + 1, trials=MAX_TRIALS)
SCORES = {
    "seed": 0,
    "attractive": 5,
    "delayed": 1,
    "weak": 0,
    "weak-2": 0,
    "weak-3": 0,
    "weak-4": 0,
    "plateau-1": 5,
    "plateau-2": 5,
    "plateau-3": 5,
    "plateau-4": 5,
    "plateau-5": 5,
    "winner": 10,
}


def evaluate(candidate):
    return SCORES[candidate]


TASK = meta.Task(
    evaluator=evaluate,
    objectives=(meta.Maximize("score"),),
    budget=RUN_BUDGET,
)

<a id="3-define-how-a-revision-is-proposed"></a>

## 3. Define how revisions are proposed

The **parent** is the version we are revising. This simulated proposer tries
`attractive` the first time it receives `seed`, then `delayed`, then the
weak versions. Revising `attractive` only produces more versions scoring 5.
Revising `delayed` produces `winner`.

`proposer()` creates a fresh function with its own attempt counts. Call it
once per run so earlier attempts cannot affect a later comparison. Its
`context` argument is unused; this example needs no feedback.

In [ ]:
def proposer():
    counts: dict[str, int] = {}

    def propose(parent, *, context):
        index = counts.get(parent, 0)
        counts[parent] = index + 1
        if parent == "seed":
            return (
                "attractive",
                "delayed",
                "weak",
                "weak-2",
                "weak-3",
                "weak-4",
            )[index]
        if parent == "delayed":
            return "winner"
        if parent == "attractive":
            return f"plateau-{index + 1}"
        return parent

    return propose

## 4. Try a handwritten loop

Evaluate the starting version, try six revisions, and keep a revision only
when its score is strictly better. This is Greedy's basic parent-selection rule.

In [ ]:
manual_propose = proposer()
manual_best = "seed"
manual_score = evaluate(manual_best)
manual_evaluations = 1
for _ in range(MAX_TRIALS):
    candidate = manual_propose(manual_best, context=None)
    score = evaluate(candidate)
    manual_evaluations += 1
    if score > manual_score:
        manual_best, manual_score = candidate, score
print(f"Handwritten: {manual_best}, score={manual_score}, evaluations={manual_evaluations}")
# Output:
# Handwritten: attractive, score=5, evaluations=7

This loop is reasonable for a disposable experiment. You can also write
different search rules yourself. The next cell adds recorded attempts and lets
us swap the search policy while keeping the rest of the experiment fixed.

<a id="4-run-with-the-default-search"></a>
<a id="change-the-search-declaration"></a>
<a id="declare-the-search-policy"></a>
<a id="5-make-search-configurable"></a>
<a id="change-only-search"></a>
<a id="run-and-compare"></a>
<a id="6-see-what-changed"></a>

## 5. Compare Greedy and TreeSearch

`improve()` uses Greedy. To choose another policy, declare an `Experiment`
and pass it to `run()`. **Greedy** revises the best version so far.
**TreeSearch** also explores other branches. Both inherit the allowance in
`TASK`; the starting version, evaluator, proposal rules, and random seed stay
the same, with fresh proposer state for each run.

The output reads **parent → revision: score**. `inspect().decisions` connects
each recorded parent choice to its attempt. `trials()` includes the starting
evaluation; `best()` returns the selected version and `usage()` reports the work.

In [ ]:
search_runs = {}
for name, policy in (
    ("Greedy", meta.Greedy()), ("TreeSearch", meta.policies.TreeSearch()),
):
    run = meta.run(meta.Experiment(
        task=TASK, seed="seed", proposer=proposer(),
        search=policy, random_seed=ROOT_SEED,
    ))
    search_runs[name] = run
    print(f"{name}: seed score={run.trials()[0].metrics['score']}")
    for choice in run.inspect().decisions:
        if choice.kind == "trial":
            attempt = choice.trial
            print(f"  {choice.parents[0].value} → "
                  f"{attempt.artifact.value}: {attempt.metrics['score']}")
    summary = run.summary()
    print(f"  Selected: {run.best().value}, score={summary.primary_score}, "
          f"evaluations={summary.evaluation_count}, proposals={run.usage().trials}")
# Output:
# Greedy: seed score=0
#   seed → attractive: 5
#   attractive → plateau-1: 5
#   attractive → plateau-2: 5
#   attractive → plateau-3: 5
#   attractive → plateau-4: 5
#   attractive → plateau-5: 5
#   Selected: attractive, score=5, evaluations=7, proposals=6
# TreeSearch: seed score=0
#   seed → attractive: 5
#   seed → delayed: 1
#   attractive → plateau-1: 5
#   attractive → plateau-2: 5
#   seed → weak: 0
#   delayed → winner: 10
#   Selected: winner, score=10, evaluations=7, proposals=6

Greedy keeps revising `attractive` and gets no further improvement. TreeSearch
tries `delayed`, then later revises it into `winner`. A weaker version can
therefore be worth revisiting. This example illustrates that possibility;
which search works well depends on your task and allowance.

Each run used six proposals and seven evaluations. The handwritten loop's
seven evaluations are separate work. These first runs keep history in memory.

## Change and predict

In cell 2, change `MAX_TRIALS = 6` to `MAX_TRIALS = 5`, then run cells 2–5.
Both searches now select `attractive` at **5**, using six evaluations each.
TreeSearch needs its sixth proposal to reach `winner` here.

Set `MAX_TRIALS = 0` to measure only `seed`: score 0, one evaluation.
Restore 6 and rerun cells 2–5 before continuing.

For your own task, replace the starting value, `evaluate`, and `proposer`.
Hold those pieces and the allowance fixed while comparing search choices.

## 6. Inspect a tie and a failed attempt

The run records unsuccessful attempts too. Start at `attractive`: its revision
`plateau-1` scores the same 5, so the starting version remains selected.

Then simulate an evaluator error. The attempted version is still recorded,
with a failure message and no measurements. The successful starting evaluation
remains selected. These are two separate, one-proposal runs.

In [ ]:
search_tie = meta.improve(
    seed="attractive", proposer=proposer(), evaluator=evaluate, trials=1,
)
print("Tie scores:", [attempt.metrics["score"] for attempt in search_tie.trials()])
print("Still selected:", search_tie.best().value)
print("Tie work:", search_tie.usage().trials, "proposal,",
      search_tie.usage().evaluations, "evaluations")

def broken_evaluation(candidate):
    if candidate != "seed":
        raise RuntimeError("Example evaluator could not run.")
    return evaluate(candidate)

search_failure = meta.improve(
    seed="seed", proposer=proposer(), evaluator=broken_evaluation, trials=1,
)
search_failed_attempt = search_failure.trials()[1]
print("Failed version:", search_failed_attempt.artifact.value)
print("Failure state:", search_failed_attempt.state)
print("Failure:", search_failed_attempt.failure.kind, search_failed_attempt.failure.message)
print("Measurements:", dict(search_failed_attempt.metrics))
print("Still selected:", search_failure.best().value)
print("Work:", search_failure.usage().trials, "proposal,",
      search_failure.usage().evaluations, "evaluations")
# Output:
# Tie scores: [5, 5]
# Still selected: attractive
# Tie work: 1 proposal, 2 evaluations
# Failed version: attractive
# Failure state: evaluation_failed
# Failure: evaluator_failure Example evaluator could not run.
# Measurements: {}
# Still selected: seed
# Work: 1 proposal, 2 evaluations

An incorrect answer can receive a low score. An evaluator that could not run
has no score to compare. Here the failed evaluation still counts as work:
one proposal and two evaluations, including the starting version.

<a id="save-the-search"></a>

## 7. Save a new run

To keep history, open `Storage.durable()` and pass it as `storage=`.
This executes a fresh TreeSearch run on disk; it does not save the earlier
in-memory runs.

A new folder preserves previous saves when you rerun the cell.
`last-search-run.txt` records that folder's path so we can find it after a
kernel restart. Capture the summary while storage is open.

In [ ]:
from pathlib import Path
from uuid import uuid4

search_directory = Path("runs") / f"search-{uuid4().hex[:12]}"
search_directory.mkdir(parents=True)
with meta.Storage.durable(search_directory / "history") as search_storage:
    saved_search = meta.run(meta.Experiment(
        task=TASK, seed="seed", proposer=proposer(),
        search=meta.policies.TreeSearch(), random_seed=ROOT_SEED,
    ), storage=search_storage)
    saved_search_summary = saved_search.summary()

Path("last-search-run.txt").write_text(str(search_directory), encoding="utf-8")
print("Saved score:", saved_search_summary.primary_score)
print("Saved evaluations:", saved_search_summary.evaluation_count)
# Output:
# Saved score: 10
# Saved evaluations: 7

## 8. Restart the kernel and reopen

**Restart the kernel**, then run this cell alone in the same working directory.
It needs only Meta-Evolve and the saved files, including the path note.
No proposer, evaluator, or earlier Python variables are needed.

In [ ]:
from pathlib import Path
import meta_evolve as meta

search_directory = Path(Path("last-search-run.txt").read_text(encoding="utf-8"))
with meta.open_run(search_directory / "history") as reopened_search:
    for search_attempt in reopened_search.trials():
        print(search_attempt.artifact.value, search_attempt.metrics["score"])
    reopened_summary = reopened_search.summary()
    search_selected = reopened_search.best().value
    print("Selected path:", " → ".join(reopened_summary.winning_lineage))
    print("Retained evaluations:", reopened_summary.evaluation_count)
# Output:
# seed 0
# attractive 5
# delayed 1
# plateau-1 5
# plateau-2 5
# weak 0
# winner 10
# Selected path: seed → delayed → winner
# Retained evaluations: 7

The seven measurements are still there. **Reopening reads history**; it does
not propose revisions or spend more evaluations. Continuing unfinished work
is a separate capability called **resuming**. Saving these local functions
does not make them resumable.

Read results inside the `with` block while the history is open. `search_selected`
is an ordinary string you can keep after it closes. The
[durability reference](https://sentient-xyz.github.io/meta-evolve-docs/guides/durability/#resume) covers continuation.

Keep the saved folder and path note on persistent disk if your notebook service
discards local files when its runtime is deleted.
[Save and reopen a run](https://sentient-xyz.github.io/meta-evolve-docs/learn/06-persist-run/) covers choosing an older run and
exporting a selected program you can use elsewhere.

## Go further

- [Improve a skill](https://sentient-xyz.github.io/meta-evolve-docs/guides/skill-composition/) or
  [a playbook](https://sentient-xyz.github.io/meta-evolve-docs/guides/playbook-improvement/) using recorded mistakes.
- [Give the proposer feedback](https://sentient-xyz.github.io/meta-evolve-docs/learn/05-use-experience/) when revisions should use
  the mistakes recorded by your evaluator.
- [Search presets](https://sentient-xyz.github.io/meta-evolve-docs/guides/runs-and-results/#search-presets) describes
  BestOfN, population, and specialist search.
- The [full policy comparison](https://github.com/sentient-xyz/meta-evolve/tree/main/examples/learn/policy_comparison)
  contains the canonical fixture and the optional five-policy command.

[Inspect results and failures](https://sentient-xyz.github.io/meta-evolve-docs/learn/03-inspect-results/) ·
[Apply the pattern elsewhere](https://sentient-xyz.github.io/meta-evolve-docs/examples/)